# 配套实践 10-02：训练任务相关的多模态 Context

本练习训练一个小型 Transformer。每个样本同时给出视觉证据、触觉证据和语言目标：目标 0 要求根据视觉作判断，目标 1 要求根据触觉作判断。模型必须先读懂目标，再选择相关模态。训练后通过目标交换和模态消融检查它是否真的形成了任务相关 Context。依赖：PyTorch、NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/10-multimodal-context-model/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import warnings  # 控制 PyTorch 教学环境中的非关键提示
import numpy as np  # 汇总训练曲线和消融结果
import torch  # 构造多模态张量并训练 Context Transformer
from torch import nn  # 使用 Transformer、线性投影和分类损失
import matplotlib.pyplot as plt  # 绘制任务规则、学习曲线和干预结果
warnings.filterwarnings("ignore", message="enable_nested_tensor")  # 隐藏不影响本实验的嵌套张量提示
torch.set_num_threads(2)  # 限制轻量实验的 CPU 线程避免额外开销
torch.manual_seed(10)  # 固定模型初始化与批次打乱顺序
np.random.seed(10)  # 固定 NumPy 侧的可视化过程
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 同一观测，目标决定应该读取哪个模态

视觉值与触觉值都可能为正或负。目标为 Visual 时，标签只由视觉值是否大于零决定；目标为 Touch 时，标签只由触觉值决定。另一模态是同时存在但与当前任务无关的干扰证据。

In [ ]:
def make_dataset(sample_count, random_seed):  # 定义生成多模态目标条件数据的函数
    generator = torch.Generator().manual_seed(random_seed)  # 为当前数据集建立独立随机生成器
    visual_values = torch.randn(sample_count, generator=generator)  # 生成可正可负的视觉证据
    touch_values = torch.randn(sample_count, generator=generator)  # 生成与视觉独立的触觉证据
    goals = torch.randint(0, 2, (sample_count,), generator=generator)  # 随机选择视觉目标或触觉目标
    labels = torch.where(goals == 0, visual_values > 0, touch_values > 0).long()  # 根据目标选择相应模态产生标签
    cls_values = torch.zeros(sample_count)  # 为汇总输出准备数值为零的 CLS token
    token_values = torch.stack([cls_values, visual_values, touch_values, goals.float()], dim=1).unsqueeze(-1)  # 按 CLS、视觉、触觉、目标排列四个 token
    return token_values, goals, labels, visual_values, touch_values  # 返回模型输入、目标、标签和原始证据
train_tokens, train_goals, train_labels, train_visual, train_touch = make_dataset(2000, 1)  # 生成用于参数更新的训练集
test_tokens, test_goals, test_labels, test_visual, test_touch = make_dataset(600, 2)  # 生成独立未见测试集
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharex=True, sharey=True)  # 创建两个语言目标下的数据分布图
for goal_value, axis, title in zip([0, 1], axes, ["Goal: use vision", "Goal: use touch"]):  # 依次显示视觉目标和触觉目标
    selected = test_goals == goal_value  # 找出当前目标对应的测试样本
    colors = np.where(test_labels[selected].numpy() == 1, "#2563eb", "#cbd5e1")  # 用颜色区分两个真实类别
    axis.scatter(test_visual[selected], test_touch[selected], c=colors, s=20, alpha=0.75)  # 在视觉与触觉平面显示样本
    axis.axvline(0.0, color="#ea580c", linestyle="--")  # 标出视觉证据的零阈值
    axis.axhline(0.0, color="#7c3aed", linestyle="--")  # 标出触觉证据的零阈值
    axis.set(title=title, xlabel="Visual evidence", ylabel="Touch evidence")  # 标注两个模态和当前任务目标
fig.suptitle("The goal selects which modality defines the label")  # 强调语言目标改变决策边界
fig.tight_layout()  # 调整两个坐标轴间距
plt.show()  # 显示目标条件下的两种分类规则

**怎样理解结果：** 蓝点表示标签 1。左图中颜色沿橙色竖线分开，触觉纵坐标不决定标签；右图中颜色沿紫色横线分开，视觉横坐标变成干扰项。因此同一个视觉—触觉组合在目标改变后可能得到不同答案，单纯拼接传感器却忽略目标无法稳定完成任务。

## 2. 用 CLS token 汇总任务相关 Context

四个标量先投影到共同维度，再加上可学习的 token 类型 embedding。两层 Transformer 让 CLS 读取视觉、触觉和目标，最后分类头只接收 CLS 输出。这个实验刻意保持很小，以便把注意力放在接口和对照上。

In [ ]:
class TinyContextModel(nn.Module):  # 定义用于多模态条件判断的小型 Context Model
    def __init__(self):  # 初始化共同投影、类型标识和 Transformer
        super().__init__()  # 初始化 PyTorch 模型基类
        self.value_projection = nn.Linear(1, 20)  # 把每个标量内容投影到二十维共同空间
        self.type_embedding = nn.Parameter(torch.randn(1, 4, 20) * 0.02)  # 为 CLS、视觉、触觉和目标建立来源标识
        encoder_layer = nn.TransformerEncoderLayer(d_model=20, nhead=4, dim_feedforward=40, dropout=0.0, batch_first=True, norm_first=True)  # 建立四头 Pre-Norm Transformer Block
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)  # 使用两层 Self-Attention 融合四类 token
        self.classifier = nn.Linear(20, 2)  # 把 CLS Context 映射到两个任务类别
    def forward(self, token_values, key_padding_mask=None):  # 定义支持模态缺失 mask 的前向计算
        tokens = self.value_projection(token_values) + self.type_embedding  # 合并内容表示与 token 类型身份
        context_tokens = self.encoder(tokens, src_key_padding_mask=key_padding_mask)  # 在有效多模态 token 之间交换信息
        return self.classifier(context_tokens[:, 0])  # 只使用第一个 CLS token 产生任务判断
model = TinyContextModel()  # 创建待训练的 Context Model
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)  # 使用 Adam 优化全部可学习参数
loss_function = nn.CrossEntropyLoss()  # 使用交叉熵监督二分类输出
loss_history = []  # 保存每轮训练损失
accuracy_history = []  # 保存每轮未见数据准确率
for epoch_index in range(30):  # 重复三十轮小批量训练
    shuffled_indices = torch.randperm(len(train_tokens))  # 在每轮开始时打乱训练样本
    batch_losses = []  # 暂存当前轮各批次损失
    for start_index in range(0, len(train_tokens), 200):  # 每次使用二百个样本更新模型
        batch_indices = shuffled_indices[start_index:start_index + 200]  # 取出当前小批量索引
        logits = model(train_tokens[batch_indices])  # 让 Context Model 融合当前批次的多模态 token
        loss = loss_function(logits, train_labels[batch_indices])  # 比较模型判断与目标条件标签
        optimizer.zero_grad()  # 清除上一个批次残留的梯度
        loss.backward()  # 反向传播计算投影、Attention 和分类头梯度
        optimizer.step()  # 更新 Context Model 的全部参数
        batch_losses.append(float(loss.detach()))  # 保存当前批次损失用于轮次汇总
    model.eval()  # 切换到评估模式计算未见数据表现
    with torch.no_grad():  # 关闭评估过程的梯度记录
        test_predictions = model(test_tokens).argmax(dim=1)  # 得到测试集上的离散预测
        test_accuracy = (test_predictions == test_labels).float().mean().item()  # 计算未见数据准确率
    model.train()  # 恢复训练模式继续下一轮参数更新
    loss_history.append(float(np.mean(batch_losses)))  # 记录当前轮平均训练损失
    accuracy_history.append(test_accuracy)  # 记录当前轮测试准确率
epochs = np.arange(1, len(loss_history) + 1)  # 建立训练轮次横轴
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))  # 创建损失和准确率两幅学习曲线
axes[0].plot(epochs, loss_history, color="#2563eb")  # 绘制训练交叉熵随轮次的变化
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy")  # 标注训练损失坐标含义
axes[1].plot(epochs, accuracy_history, color="#16a34a")  # 绘制未见数据准确率随轮次的变化
axes[1].axhline(0.5, color="#dc2626", linestyle="--", label="Random guess")  # 标出二分类随机猜测水平
axes[1].set(title="Accuracy on unseen samples", xlabel="Epoch", ylabel="Accuracy", ylim=(0.45, 1.02))  # 设置准确率显示范围
axes[1].legend()  # 显示随机猜测参考线名称
for axis in axes:  # 为两幅学习曲线统一添加网格
    axis.grid(alpha=0.2)  # 使用淡网格帮助读取数值变化
fig.tight_layout()  # 调整两幅图之间的间距
plt.show()  # 显示 Context Model 的学习过程

**怎样理解结果：** 随着训练进行，损失下降而未见数据准确率接近 1，说明小型 Transformer 有能力表示“先读目标，再选择相应模态”的规则。但高准确率还不能证明它一定按预期使用了目标和传感器，下一步必须主动改变输入。

## 3. 交换目标并移除模态

我们保留原来的真实标签，分别执行四种干预：交换语言目标、屏蔽视觉、屏蔽触觉和屏蔽目标。如果 Context 真正使用了这些输入，表现应以符合任务结构的方式下降。

In [ ]:
def evaluate_accuracy(input_tokens, key_padding_mask=None):  # 定义在给定输入干预下计算准确率的函数
    model.eval()  # 把模型切换到确定性的评估模式
    with torch.no_grad():  # 关闭干预评估过程的梯度记录
        predictions = model(input_tokens, key_padding_mask).argmax(dim=1)  # 得到干预输入下的模型类别
    return float((predictions == test_labels).float().mean())  # 始终与原始任务标签比较并返回准确率
full_accuracy = evaluate_accuracy(test_tokens)  # 计算未修改输入下的基准表现
swapped_goal_tokens = test_tokens.clone()  # 复制测试 token 以免修改原始数据
swapped_goal_tokens[:, 3, 0] = 1.0 - swapped_goal_tokens[:, 3, 0]  # 把视觉目标与触觉目标相互交换
swapped_goal_accuracy = evaluate_accuracy(swapped_goal_tokens)  # 检查目标改变后原标签还能否保持
def mask_one_token(token_index):  # 定义屏蔽一种模态 token 的评估函数
    mask = torch.zeros(len(test_tokens), 4, dtype=torch.bool)  # 建立四个 token 均可读取的初始 mask
    mask[:, token_index] = True  # 把指定模态标记为所有样本都不可作为 Key
    return evaluate_accuracy(test_tokens, mask)  # 返回该模态缺失时的测试准确率
vision_missing_accuracy = mask_one_token(1)  # 屏蔽视觉证据并计算表现
touch_missing_accuracy = mask_one_token(2)  # 屏蔽触觉证据并计算表现
goal_missing_accuracy = mask_one_token(3)  # 屏蔽语言目标并计算表现
condition_names = ["Full input", "Swap goal", "No vision", "No touch", "No goal"]  # 定义五种输入条件名称
condition_accuracies = [full_accuracy, swapped_goal_accuracy, vision_missing_accuracy, touch_missing_accuracy, goal_missing_accuracy]  # 汇总五种条件准确率
bar_colors = ["#2563eb", "#ea580c", "#94a3b8", "#94a3b8", "#7c3aed"]  # 为基准与不同干预分配颜色
fig, axis = plt.subplots(figsize=(9, 4.0))  # 创建输入干预结果柱状图
bars = axis.bar(condition_names, condition_accuracies, color=bar_colors)  # 绘制五种条件下的测试准确率
axis.axhline(0.5, color="#dc2626", linestyle="--", label="Random guess")  # 标出随机猜测水平
axis.set(title="Input interventions reveal what the Context uses", ylabel="Accuracy", ylim=(0.4, 1.05))  # 标注干预评估含义与范围
for bar, accuracy in zip(bars, condition_accuracies):  # 依次读取每个柱子及其准确率
    axis.text(bar.get_x() + bar.get_width() / 2, accuracy + 0.015, f"{accuracy:.2f}", ha="center")  # 在柱子上方写出两位小数结果
axis.legend()  # 显示随机猜测参考线图例
fig.tight_layout()  # 调整柱状图边距
plt.show()  # 显示目标交换和模态缺失的影响

**怎样理解结果：** 完整输入接近满分；交换目标后，模型会按照新目标选择另一模态，因此与原标签比较时接近随机水平。移除视觉或触觉时，仍有约一半样本使用未缺失的另一模态并可正确判断，另一半只能猜测，所以总体约为 75%。移除目标后，模型不知道两种证据中哪一个定义当前任务，表现也明显下降。

**本练习的结论：** Context 是否“任务相关”不能只由完整输入准确率证明。目标交换、模态消融和时间打乱能够把模型依赖的信息暴露出来；真实机器人课程后续也会沿用这种干预式检查。